In [1]:
# Importing the necessary libraries
import transformers  # Hugging Face library for working with transformer models
import torch         # PyTorch library for tensor operations and model handling

class MedicalChatbotPipeline:
    """
    A class-based implementation for interacting with the Hugging Face Transformers pipeline.
    This chatbot is tailored for the HPAI-BSC/Llama3.1-Aloe-Beta-70B medical language model.
    """

    def __init__(self, model_id="HPAI-BSC/Llama3.1-Aloe-Beta-70B"):
        """
        Initializes the MedicalChatbotPipeline class with the specified model.

        Args:
            model_id (str): The identifier of the Hugging Face model to load.
        """
        # Define the model ID to be used
        self.model_id = model_id

        # Initialize the text-generation pipeline using the specified model
        # 'torch_dtype=torch.bfloat16' reduces memory usage by using lower precision
        # 'device_map="auto"' automatically selects the best available device (CPU or GPU)
        self.pipeline = transformers.pipeline(
            "text-generation",                   # Specifies the task type (text generation)
            model=self.model_id,                 # Specifies the model to load
            model_kwargs={"torch_dtype": torch.bfloat16},  # Optimized precision for efficiency
            device_map="auto",                   # Automatically assign the device (CPU/GPU)
        )

    def generate_response(self, user_message, max_tokens=256, temperature=0.6, top_p=0.9):
        """
        Generates a response from the medical chatbot.

        Args:
            user_message (str): The user's input query.
            max_tokens (int): Maximum number of tokens to generate in the response.
            temperature (float): Controls the randomness of the response.
            top_p (float): Controls the diversity of the response using nucleus sampling.

        Returns:
            str: The generated response from the chatbot.
        """
        # Define the conversation messages in a structured format
        # 'role' indicates the speaker, while 'content' contains their message
        messages = [
            {
                "role": "system",
                "content": (
                    "You are an expert medical assistant named Aloe, developed by the High "
                    "Performance Artificial Intelligence Group at Barcelona Supercomputing Center (BSC). "
                    "You are to be a helpful, respectful, and honest assistant."
                ),
            },
            {"role": "user", "content": user_message},  # Add the user's query as a message
        ]

        # Convert the conversation into a structured input format for the model
        # 'apply_chat_template' prepares the input based on the conversation format
        # 'tokenize=False' ensures that the input remains as plain text (not tokenized yet)
        # 'add_generation_prompt=True' signals the model to start generating a response
        prompt = self.pipeline.tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=True
        )

        # Define termination tokens to signal the end of generation
        # 'eos_token_id' is the standard end-of-sequence token for the model
        # '<|eot_id|>' is a custom end-of-text token for this specific chatbot
        terminators = [
            self.pipeline.tokenizer.eos_token_id,  # Standard end-of-sequence token
            self.pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>"),  # Custom token ID
        ]

        # Generate the chatbot's response
        outputs = self.pipeline(
            prompt,                       # Input prompt for the model
            max_new_tokens=max_tokens,    # Limit the maximum length of the generated text
            eos_token_id=terminators,     # End-of-sequence token(s) to stop generation
            do_sample=True,               # Enable probabilistic sampling for diverse responses
            temperature=temperature,      # Controls randomness (higher = more random)
            top_p=top_p,                  # Enables nucleus sampling for response diversity
        )

        # Extract and clean the generated response
        # Slice off the prompt length to retrieve only the model's response
        response = outputs[0]["generated_text"][len(prompt):].strip()
        
        # Return the cleaned response
        return response





In [2]:
#AutoModelForCausalLM

# Importing the necessary libraries
from transformers import AutoTokenizer, AutoModelForCausalLM  # For tokenizer and model handling
import torch  # PyTorch library for tensor manipulation and device management

class MedicalChatbotAutoModel:
    """
    A class-based implementation for interacting with the Hugging Face AutoModelForCausalLM approach.
    This chatbot is tailored for the HPAI-BSC/Llama3.1-Aloe-Beta-70B medical language model.
    """

    def __init__(self, model_id="HPAI-BSC/Llama3.1-Aloe-Beta-70B"):
        """
        Initializes the MedicalChatbotAutoModel class with the specified model.

        Args:
            model_id (str): The identifier of the Hugging Face model to load.
        """
        # Define the model ID to be used
        self.model_id = model_id

        # Load the tokenizer associated with the model
        # The tokenizer is responsible for converting text to tokens and vice versa
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)

        # Load the model with specified configurations
        # 'torch_dtype=torch.bfloat16' reduces memory usage with lower precision
        # 'device_map="auto"' automatically selects the best available device (CPU/GPU)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_id,
            torch_dtype=torch.bfloat16,  # Optimized precision for efficiency
            device_map="auto",           # Automatically assign the device (CPU/GPU)
        )

    def generate_response(self, user_message, max_tokens=256, temperature=0.6, top_p=0.9):
        """
        Generates a response from the medical chatbot.

        Args:
            user_message (str): The user's input query.
            max_tokens (int): Maximum number of tokens to generate in the response.
            temperature (float): Controls the randomness of the response.
            top_p (float): Controls the diversity of the response using nucleus sampling.

        Returns:
            str: The generated response from the chatbot.
        """
        # Define the conversation messages in a structured format
        # 'role' indicates the speaker, while 'content' contains their message
        messages = [
            {
                "role": "system",
                "content": (
                    "You are an expert medical assistant named Aloe, developed by the High "
                    "Performance Artificial Intelligence Group at Barcelona Supercomputing Center (BSC). "
                    "You are to be a helpful, respectful, and honest assistant."
                ),
            },
            {"role": "user", "content": user_message},  # Add the user's query as a message
        ]

        # Convert the conversation into input IDs using the tokenizer
        # 'apply_chat_template' formats the conversation for the model
        # 'add_generation_prompt=True' signals the model to start generating a response
        # 'return_tensors="pt"' ensures the output is in PyTorch tensor format
        input_ids = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt"  # Return as PyTorch tensors for model compatibility
        ).to(self.model.device)  # Move the input IDs to the model's device (CPU/GPU)

        # Define termination tokens to signal the end of generation
        # 'eos_token_id' is the standard end-of-sequence token for the model
        # '<|eot_id|>' is a custom end-of-text token for this specific chatbot
        terminators = [
            self.tokenizer.eos_token_id,  # Standard end-of-sequence token
            self.tokenizer.convert_tokens_to_ids("<|eot_id|>"),  # Custom token ID
        ]

        # Generate the chatbot's response using the model's generate method
        outputs = self.model.generate(
            input_ids,                  # The formatted conversation as input IDs
            max_new_tokens=max_tokens,  # Limit the maximum length of the generated text
            eos_token_id=terminators,   # End-of-sequence token(s) to stop generation
            do_sample=True,             # Enable probabilistic sampling for diverse responses
            temperature=temperature,    # Controls randomness (higher = more random)
            top_p=top_p,                # Enables nucleus sampling for response diversity
        )

        # Extract the generated response by slicing the output tensor
        # The slicing removes the input prompt, leaving only the generated text
        response_ids = outputs[0][input_ids.shape[-1]:]  # Skip input prompt tokens
        response = self.tokenizer.decode(
            response_ids, skip_special_tokens=True  # Convert token IDs to text
        ).strip()  # Remove any leading/trailing whitespace for cleanliness

        # Return the cleaned response
        return response




# Comparison

| Feature                     | Transformers Pipeline Approach         | AutoModelForCausalLM Approach            |
|-----------------------------|-----------------------------------------|------------------------------------------|
| **Ease of Use**             | High (Beginner-friendly)               | Moderate to Low (Requires more effort)   |
| **Customization**           | Limited                                | High                                     |
| **Performance Optimization**| Slight Overhead                        | Slightly Faster                          |
| **Flexibility**             | Moderate (Pre-defined functionality)   | High (Full control over model usage)     |
| **Learning Curve**          | Low                                    | High                                     |


In [3]:
# Import the necessary libraries and classes from separate files for each approach.
#from approach1_pipeline import MedicalChatbotPipeline  # Import the pipeline-based chatbot class
#from approach2_automodel import MedicalChatbotAutoModel  # Import the automodel-based chatbot class


def get_user_query():
    """
    Allows the user to select from predefined medical queries or enter a custom query.
    Returns the selected query as a string.
    """
    # List of common medical queries for testing purposes
    common_queries = [
        "What are the symptoms of the flu?",
        "Can you tell me about the side effects of ibuprofen?",
        "What should I do if I have a fever?",
        "How can I lower my blood pressure naturally?",
        "What are the early signs of diabetes?",
        "What foods should I eat to improve digestion?",
        "Can you explain the difference between a cold and COVID-19?",
    ]

    # Display options for the user
    print("\nChoose a predefined medical query or enter a custom one:")
    for i, query in enumerate(common_queries, start=1):
        print(f"{i}. {query}")  # Print each predefined query with a number

    print(f"{len(common_queries) + 1}. Enter a custom query")  # Option to enter a custom query

    # Get the user's choice
    while True:
        choice = input("\nEnter the number of your choice: ").strip()
        
        # Ensure the choice is a valid number within the range
        if choice.isdigit():
            choice = int(choice)
            if 1 <= choice <= len(common_queries):  # User picked a predefined query
                return common_queries[choice - 1]
            elif choice == len(common_queries) + 1:  # User wants to enter a custom query
                return input("\nEnter your medical question: ").strip()
        
        print("Invalid choice. Please enter a valid number.")  # Error handling for invalid input



In [4]:

def main_pipeline():
    """
    Runs the MedicalChatbotPipeline approach with a user-selected query.
    """
    print("\nRunning MedicalChatbotPipeline approach...\n")  # Indicate the approach being used

    # Get the user's chosen medical query
    user_query = get_user_query()

    # Initialize the chatbot using the pipeline approach
    chatbot = MedicalChatbotPipeline()

    # Generate a response using the chatbot
    response = chatbot.generate_response(user_query)

    # Display the chatbot's response
    print("\nResponse (Pipeline Approach):", response)


In [5]:

def main_automodel():
    """
    Runs the MedicalChatbotAutoModel approach with a user-selected query.
    """
    print("\nRunning MedicalChatbotAutoModel approach...\n")  # Indicate the approach being used

    # Get the user's chosen medical query
    user_query = get_user_query()

    # Initialize the chatbot using the automodel approach
    chatbot = MedicalChatbotAutoModel()
    
    # Generate a response using the chatbot
    response = chatbot.generate_response(user_query)

    # Display the chatbot's response
    print("\nResponse (AutoModel Approach):", response)



In [6]:

if __name__ == "__main__":
    """
    Main entry point of the script. Allows the user to select the chatbot approach
    and provides options to test different medical queries.
    """
    # Ask the user which chatbot approach they want to use
    print("\nSelect the chatbot approach to run:")
    print("1. MedicalChatbotPipeline (Pipeline Approach)")
    print("2. MedicalChatbotAutoModel (AutoModel Approach)")

    # Get user input for selecting the approach
    while True:
        choice = input("\nEnter 1 or 2: ").strip()

        if choice == "1":
            main_pipeline()  # Run the pipeline approach
            break
        elif choice == "2":
            main_automodel()  # Run the automodel approach
            break
        else:
            print("Invalid choice. Please enter 1 or 2.")  # Error handling for invalid input



Select the chatbot approach to run:
1. MedicalChatbotPipeline (Pipeline Approach)
2. MedicalChatbotAutoModel (AutoModel Approach)

Running MedicalChatbotAutoModel approach...


Choose a predefined medical query or enter a custom one:
1. What are the symptoms of the flu?
2. Can you tell me about the side effects of ibuprofen?
3. What should I do if I have a fever?
4. How can I lower my blood pressure naturally?
5. What are the early signs of diabetes?
6. What foods should I eat to improve digestion?
7. Can you explain the difference between a cold and COVID-19?
8. Enter a custom query


ImportError: Using `low_cpu_mem_usage=True` or a `device_map` requires Accelerate: `pip install 'accelerate>=0.26.0'`